# Notebook 02: Consolidación y creación de variables
**Proyecto:** Predicción de Insatisfacción del Cliente — Olist (Grupo C)
**Objetivo:** unir las tablas traducidas en una única tabla con **una fila por pedido**, que contenga
la variable objetivo y las variables explicativas propuestas (retraso, flete, historial del vendedor, etc.).

In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)

RUTA_PROC = '../data/procesado'

# Columnas de fecha de cada tabla (se convierten a tipo fecha al leer)
fechas_orders = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                 'order_delivered_customer_date', 'order_estimated_delivery_date']

orders    = pd.read_csv(os.path.join(RUTA_PROC, 'orders_es.csv'), parse_dates=fechas_orders)
items     = pd.read_csv(os.path.join(RUTA_PROC, 'items_es.csv'), parse_dates=['shipping_limit_date'])
payments  = pd.read_csv(os.path.join(RUTA_PROC, 'payments_es.csv'))
reviews   = pd.read_csv(os.path.join(RUTA_PROC, 'reviews_es.csv'),
                        parse_dates=['review_creation_date', 'review_answer_timestamp'])
customers = pd.read_csv(os.path.join(RUTA_PROC, 'customers_es.csv'), dtype={'customer_zip_code_prefix': str})
products  = pd.read_csv(os.path.join(RUTA_PROC, 'products_es.csv'))

for nombre, df in [('orders', orders), ('items', items), ('payments', payments),
                   ('reviews', reviews), ('customers', customers), ('products', products)]:
    print(f'{nombre:10} -> {df.shape}')

orders     -> (99441, 9)
items      -> (112650, 7)
payments   -> (103886, 6)
reviews    -> (99224, 11)
customers  -> (99441, 7)
products   -> (32951, 10)


## Pedidos con más de una reseña
Algunos pedidos tienen varias reseñas. Para tener una sola variable objetivo por pedido,
se conserva la **reseña más reciente** (la última opinión del cliente).

In [2]:
cant_resenas = reviews['order_id'].value_counts()
print('Pedidos con más de una reseña:', (cant_resenas > 1).sum())

# ¿Cuántos de esos pedidos tienen reseñas que no coinciden en la variable objetivo?
duplicados = reviews[reviews['order_id'].duplicated(keep=False)]
conflicto = duplicados.groupby('order_id')['insatisfecho'].nunique()
print('De esos, con objetivo distinto entre reseñas:', (conflicto > 1).sum())

# Nos quedamos con la reseña más reciente de cada pedido
reviews_unicas = (reviews.sort_values('review_answer_timestamp')
                         .drop_duplicates('order_id', keep='last'))
print('Reseñas finales:', len(reviews_unicas), '| ¿una por pedido?', reviews_unicas['order_id'].is_unique)

Pedidos con más de una reseña: 547
De esos, con objetivo distinto entre reseñas: 118
Reseñas finales: 98673 | ¿una por pedido? True


## ¿Qué pedidos se incluyen?
Se analiza la insatisfacción según el estado del pedido para decidir la población de estudio.

In [3]:
tmp = orders.merge(reviews_unicas[['order_id', 'insatisfecho']], on='order_id')
display(tmp.groupby('estado_pedido')['insatisfecho']
           .agg(tasa_insatisfaccion='mean', cantidad='size')
           .sort_values('cantidad', ascending=False).round(3))

,tasa_insatisfaccion,cantidad
estado_pedido,,
entregado,0.211,95832
enviado,0.801,1032
cancelado,0.848,605
no_disponible,0.908,595
facturado,0.880,309
en_proceso,0.959,295
creado,0.667,3
aprobado,0.500,2


**Decisión:** se trabaja solo con pedidos **entregados** y con fecha de entrega registrada.
- Los pedidos no entregados (cancelados, no disponibles, enviados sin llegar) tienen 80–95% de
  insatisfacción: su resultado es obvio y no requiere un modelo.
- Las variables clave de la propuesta (días de retraso, tiempo de entrega) solo existen si el pedido llegó.
- Los entregados representan ~97% de los pedidos.

In [4]:
base = orders[(orders['estado_pedido'] == 'entregado') &
              (orders['order_delivered_customer_date'].notna())].copy()
print('Pedidos entregados con fecha de entrega:', len(base))

Pedidos entregados con fecha de entrega: 96470


## Variables de tiempo
| Variable | Cálculo | Interpretación |
|---|---|---|
| `dias_entrega_real` | entrega real − compra | Cuánto tardó en llegar |
| `dias_entrega_estimada` | fecha estimada − compra | Cuánto se le prometió al cliente |
| `dias_retraso` | entrega real − fecha estimada | **> 0 llegó tarde**, < 0 llegó antes |
| `entrega_tardia` | 1 si `dias_retraso` > 0 | Indicador binario de retraso |
| `horas_aprobacion` | aprobación del pago − compra | Demora en aprobar el pago |
| `dias_hasta_transportista` | entrega al transportista − compra | Demora del vendedor en despachar |
| `mes_compra`, `dia_semana_compra` | de la fecha de compra | Estacionalidad |

In [5]:
# Función auxiliar: diferencia entre dos fechas expresada en días (con decimales)
def dif_dias(fin, inicio):
    return (base[fin] - base[inicio]).dt.total_seconds() / 86400

base['dias_entrega_real']        = dif_dias('order_delivered_customer_date', 'order_purchase_timestamp')
base['dias_entrega_estimada']    = dif_dias('order_estimated_delivery_date', 'order_purchase_timestamp')
base['dias_retraso']             = dif_dias('order_delivered_customer_date', 'order_estimated_delivery_date')
base['entrega_tardia']           = (base['dias_retraso'] > 0).astype(int)
base['horas_aprobacion']         = dif_dias('order_approved_at', 'order_purchase_timestamp') * 24
base['dias_hasta_transportista'] = dif_dias('order_delivered_carrier_date', 'order_purchase_timestamp')
base['mes_compra']               = base['order_purchase_timestamp'].dt.month
base['dia_semana_compra']        = base['order_purchase_timestamp'].dt.dayofweek   # 0 = lunes

vars_tiempo = ['dias_entrega_real', 'dias_entrega_estimada', 'dias_retraso',
               'horas_aprobacion', 'dias_hasta_transportista']
display(base[vars_tiempo].describe().round(2).T)
print('Proporción de entregas tardías:', round(base['entrega_tardia'].mean(), 3))

C:\Users\seinb\AppData\Local\Temp\ipykernel_9692\1883412946.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  base['dia_semana_compra']        = base['order_purchase_timestamp'].dt.dayofweek   # 0 = lunes


,count,mean,std,min,25%,50%,75%,max
dias_entrega_real,96470.0,12.56,9.55,0.53,6.77,10.22,15.72,209.63
dias_entrega_estimada,96470.0,23.74,8.76,2.01,18.33,23.23,28.41,155.14
dias_retraso,96470.0,-11.18,10.18,-146.02,-16.24,-11.95,-6.39,188.98
horas_aprobacion,96456.0,10.28,20.54,0.00,0.22,0.34,14.52,741.44
dias_hasta_transportista,96469.0,3.23,3.60,-171.21,1.13,2.20,4.07,125.78


Proporción de entregas tardías: 0.081


In [6]:
# Inconsistencias de fechas: despacho al transportista antes de la compra
negativos = (base['dias_hasta_transportista'] < 0).sum()
print('Pedidos con despacho anterior a la compra:', negativos)

Pedidos con despacho anterior a la compra: 165


**Inconsistencia detectada:** 165 pedidos tienen fecha de despacho al transportista anterior a la
fecha de compra (error de registro). Se tratarán en el Notebook 04 (preprocesamiento) reemplazándolos por nulo e imputando.

## Variables de ítems y productos (agregadas por pedido)
Se une cada ítem con las características de su producto y se resumen por pedido.
La **categoría principal** y el **vendedor principal** son los del ítem más caro del pedido.

In [7]:
# Unir ítems con datos del producto
cols_prod = ['product_id', 'categoria_producto', 'product_weight_g', 'product_length_cm',
             'product_height_cm', 'product_width_cm', 'product_photos_qty', 'product_description_lenght']
items_prod = items.merge(products[cols_prod], on='product_id', how='left')
items_prod['volumen_cm3'] = (items_prod['product_length_cm'] * items_prod['product_height_cm']
                             * items_prod['product_width_cm'])

# Resumen por pedido
items_agg = items_prod.groupby('order_id').agg(
    cant_items=('order_item_id', 'count'),
    cant_productos=('product_id', 'nunique'),
    cant_vendedores=('seller_id', 'nunique'),
    precio_total=('price', 'sum'),
    flete_total=('freight_value', 'sum'),
    peso_total_g=('product_weight_g', 'sum'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    largo_descripcion_promedio=('product_description_lenght', 'mean'),
    fecha_limite_despacho=('shipping_limit_date', 'max'),
).reset_index()

# Proporción del flete sobre el precio (flete caro en relación al producto)
items_agg['ratio_flete_precio'] = items_agg['flete_total'] / items_agg['precio_total']

# Categoría y vendedor del ítem más caro de cada pedido
principal = (items_prod.sort_values(['order_id', 'price'], ascending=[True, False])
                       .drop_duplicates('order_id')
                       [['order_id', 'categoria_producto', 'seller_id']]
                       .rename(columns={'categoria_producto': 'categoria_principal',
                                        'seller_id': 'vendedor_principal'}))
items_agg = items_agg.merge(principal, on='order_id')

print('Pedidos con ítems:', len(items_agg))
display(items_agg.describe().round(2).T)

Pedidos con ítems: 98666


,count,mean,min,25%,50%,75%,max,std
cant_items,98666.0,1.14,1.0,1.0,1.0,1.0,21.0,0.54
cant_productos,98666.0,1.04,1.0,1.0,1.0,1.0,8.0,0.23
cant_vendedores,98666.0,1.01,1.0,1.0,1.0,1.0,5.0,0.12
precio_total,98666.0,137.75,0.85,45.9,86.9,149.9,13440.0,210.65
flete_total,98666.0,22.82,0.0,13.85,17.17,24.04,1794.96,21.65
peso_total_g,98666.0,2390.03,0.0,300.0,750.0,2066.75,184400.0,4773.24
volumen_total_cm3,98666.0,17401.43,0.0,2964.0,7260.0,19872.0,1476000.0,30402.17
fotos_promedio,97277.0,2.25,1.0,1.0,2.0,3.0,20.0,1.74
largo_descripcion_promedio,97277.0,794.06,4.0,351.0,608.0,996.0,3992.0,652.99
fecha_limite_despacho,98666,2018-01-07 12:54:18.369002,2016-09-19 00:15:34,2017-09-20 02:55:30,2018-01-26 13:52:23.500000,2018-05-10 15:14:05.250000,2020-04-09 22:35:08,NaN


## Variables de pago (agregadas por pedido)
Un pedido puede pagarse con más de un medio (ej.: tarjeta + voucher). El **tipo de pago principal**
es el de mayor monto.

In [8]:
pagos_agg = payments.groupby('order_id').agg(
    cant_medios_pago=('payment_sequential', 'count'),
    cuotas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
).reset_index()

tipo_principal = (payments.sort_values(['order_id', 'payment_value'], ascending=[True, False])
                          .drop_duplicates('order_id')[['order_id', 'tipo_pago']])
pagos_agg = pagos_agg.merge(tipo_principal, on='order_id')

print('Pedidos con pagos:', len(pagos_agg))
display(pagos_agg['tipo_pago'].value_counts())

Pedidos con pagos: 99440


tipo_pago
tarjeta_credito    74975
boleto_bancario    19784
voucher             3151
tarjeta_debito      1527
no_definido            3
Name: count, dtype: int64

## Unión de todas las tablas
Se parte de la base de pedidos entregados y se agregan: reseña (variable objetivo), ítems, pagos y cliente.
Con la reseña se usa unión **interna** (`inner`): un pedido sin reseña no tiene variable objetivo.

In [9]:
df = (base
      .merge(reviews_unicas[['order_id', 'review_score', 'insatisfecho',
                             'tiene_comentario', 'largo_comentario']], on='order_id', how='inner')
      .merge(items_agg, on='order_id', how='left')
      .merge(pagos_agg, on='order_id', how='left')
      .merge(customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_region']],
             on='customer_id', how='left'))

# Demora del vendedor respecto a la fecha límite de despacho
df['dias_despacho_vs_limite'] = ((df['order_delivered_carrier_date'] - df['fecha_limite_despacho'])
                                 .dt.total_seconds() / 86400)
df['despacho_tardio'] = (df['dias_despacho_vs_limite'] > 0).astype(int)

print('Filas:', len(df), '| ¿un pedido por fila?', df['order_id'].is_unique)
print('Tasa de insatisfacción:', round(df['insatisfecho'].mean(), 3))

Filas: 95824 | ¿un pedido por fila? True
Tasa de insatisfacción: 0.211


## Historial del vendedor (sin fuga de información)
Para cada pedido se calcula el historial de su vendedor principal usando **solo información
disponible al momento de la compra**:

| Variable | Qué mide |
|---|---|
| `vendedor_pedidos_previos` | Pedidos que el vendedor recibió antes (experiencia/volumen) |
| `vendedor_tasa_retraso_previa` | % de entregas tardías entre sus pedidos ya entregados |
| `vendedor_tasa_insatisf_previa` | % de clientes insatisfechos entre sus reseñas ya respondidas |

Si el vendedor no tiene historial previo, las tasas quedan nulas (se tratan en el preprocesamiento).

In [10]:
# Necesitamos la fecha de respuesta de la reseña para saber cuándo se "conoció" la opinión
df = df.merge(reviews_unicas[['order_id', 'review_answer_timestamp']], on='order_id', how='left')

def historial_vendedor(g):
    """Calcula el historial de un vendedor para cada uno de sus pedidos,
    usando solo eventos ocurridos ANTES de la fecha de compra."""
    g = g.copy()
    compra = g['order_purchase_timestamp'].values

    # 1) Pedidos recibidos antes de esta compra
    g['vendedor_pedidos_previos'] = np.searchsorted(np.sort(compra), compra, side='left')

    # 2) Tasa de retraso: entre los pedidos ya entregados al momento de la compra
    ord_ent = g.sort_values('order_delivered_customer_date')
    tardias_acum = np.concatenate([[0], np.cumsum(ord_ent['entrega_tardia'].values)])
    k = np.searchsorted(ord_ent['order_delivered_customer_date'].values, compra, side='left')
    g['vendedor_tasa_retraso_previa'] = np.where(k > 0, tardias_acum[k] / np.maximum(k, 1), np.nan)

    # 3) Tasa de insatisfacción: entre las reseñas ya respondidas al momento de la compra
    ord_res = g.sort_values('review_answer_timestamp')
    insat_acum = np.concatenate([[0], np.cumsum(ord_res['insatisfecho'].values)])
    k = np.searchsorted(ord_res['review_answer_timestamp'].values, compra, side='left')
    g['vendedor_tasa_insatisf_previa'] = np.where(k > 0, insat_acum[k] / np.maximum(k, 1), np.nan)
    return g

# Se aplica vendedor por vendedor y se vuelve a unir
df = pd.concat([historial_vendedor(g) for _, g in df.groupby('vendedor_principal')])
df = df.sort_values('order_purchase_timestamp').reset_index(drop=True)

vars_vendedor = ['vendedor_pedidos_previos', 'vendedor_tasa_retraso_previa', 'vendedor_tasa_insatisf_previa']
display(df[vars_vendedor].describe().round(3).T)

,count,mean,std,min,25%,50%,75%,max
vendedor_pedidos_previos,95824.0,181.317,292.843,0.0,15.00,58.000,197.000,1786.0
vendedor_tasa_retraso_previa,90585.0,0.053,0.073,0.0,0.00,0.038,0.073,1.0
vendedor_tasa_insatisf_previa,89951.0,0.199,0.131,0.0,0.13,0.193,0.253,1.0


In [11]:
# Tasa de insatisfacción según el historial de insatisfacción del vendedor
tramos = pd.cut(df['vendedor_tasa_insatisf_previa'], [-0.01, 0.1, 0.2, 0.3, 0.5, 1])
display(df.groupby(tramos, observed=True)['insatisfecho']
          .agg(tasa_insatisfaccion='mean', cantidad='size').round(3))

,tasa_insatisfaccion,cantidad
vendedor_tasa_insatisf_previa,,
"(-0.01, 0.1]",0.176,15989
"(0.1, 0.2]",0.190,33198
"(0.2, 0.3]",0.229,28368
"(0.3, 0.5]",0.271,10865
"(0.5, 1.0]",0.302,1531


## Guardado de la base consolidada
Se separan las columnas en grupos. Las columnas de **fuga de información** (`review_score`,
`tiene_comentario`, `largo_comentario`, `review_answer_timestamp`) se guardan solo para el EDA
y **no deben usarse como predictores**.

In [12]:
cols_fuga = ['review_score', 'tiene_comentario', 'largo_comentario', 'review_answer_timestamp']

print('Filas:', len(df), '| Columnas:', df.shape[1], '| ¿un pedido por fila?', df['order_id'].is_unique)
print('Tasa de insatisfacción:', round(df['insatisfecho'].mean(), 3))

print('\nVariables con nulos:')
nulos = df.isna().sum()
display(nulos[nulos > 0].sort_values(ascending=False))

df.to_csv(os.path.join(RUTA_PROC, 'base_consolidada.csv'), index=False)
print('Guardado en', os.path.join(RUTA_PROC, 'base_consolidada.csv'))

Filas: 95824 | Columnas: 47 | ¿un pedido por fila? True
Tasa de insatisfacción: 0.211

Variables con nulos:


vendedor_tasa_insatisf_previa    5873
vendedor_tasa_retraso_previa     5239
largo_descripcion_promedio       1324
fotos_promedio                   1324
order_approved_at                  14
horas_aprobacion                   14
order_delivered_carrier_date        1
dias_hasta_transportista            1
cant_medios_pago                    1
valor_pagado                        1
cuotas                              1
dias_despacho_vs_limite             1
tipo_pago                           1
dtype: int64

Guardado en ../data/procesado\base_consolidada.csv


## Resumen
- Base: **95.824 pedidos entregados con reseña**, una fila por pedido, 21,1% insatisfechos.
- Se crearon variables de tiempo, de ítems/productos, de pago, de cliente y de historial del vendedor.
- Pendiente para el preprocesamiento: nulos (historial de vendedores nuevos, fotos/descripción,
  fechas faltantes), 165 fechas de despacho inconsistentes, outliers en tiempos de entrega.

**Próximo paso → Notebook 03: EDA** (inciso 1 del enunciado).